In [ ]:
# Setup
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd()
sys.path.insert(0, str(project_root))

import os
import json
import requests
from datetime import datetime
import pandas as pd

print("✓ Setup complete")

## 1. ChromaDB Storage Monitor

Check how many documents are stored, collection statistics, and storage size.

In [ ]:
# ChromaDB Monitoring
from src.vectorstore.chroma_manager import get_vectorstore

vectorstore = get_vectorstore()
stats = vectorstore.get_stats()

print("📊 ChromaDB Statistics")
print("=" * 50)
print(f"Collection: {stats.get('collection_name', 'N/A')}")
print(f"Total Chunks: {stats.get('total_documents', 0)}")
print(f"Storage Path: {stats.get('persist_directory', 'N/A')}")
print()

# Check storage size
chroma_path = Path("./data/chromadb")
if chroma_path.exists():
    total_size = sum(f.stat().st_size for f in chroma_path.rglob('*') if f.is_file())
    print(f"💾 Storage Size: {total_size / (1024*1024):.2f} MB")
    print(f"📁 Files Count: {len(list(chroma_path.rglob('*')))}")
else:
    print("⚠️ ChromaDB directory not found")

print("\n🔍 Sample stored sources:")
sources = vectorstore.list_sources()
for source in sources[:5]:
    print(f"  - {source}")

## 2. Embeddings Monitor

Check embeddings model status and test embedding generation.

In [ ]:
# Embeddings Monitoring
from src.embeddings.embeddings_manager import get_embeddings

embeddings_mgr = get_embeddings()

print("🧠 Embeddings Model Status")
print("=" * 50)
print(f"Model: hkunlp/instructor-large")
print(f"Device: CPU")

# Test embedding generation
test_text = "Test semiconductor validation"
embedding = embeddings_mgr.embed_query(test_text)
print(f"\n✓ Embedding Test: Generated {len(embedding)} dimensions")
print(f"Sample values: {embedding[:5]}")

# Check model cache
cache_path = Path("./models/instructor-large")
if cache_path.exists():
    cache_size = sum(f.stat().st_size for f in cache_path.rglob('*') if f.is_file())
    print(f"\n💾 Model Cache: {cache_size / (1024*1024*1024):.2f} GB")
else:
    print("\n⚠️ Model cache not found (using default location)")

## 3. Ollama LLM Monitor

Check Ollama service status, loaded models, and track usage.

In [ ]:
# Ollama Monitoring
import subprocess

print("🤖 Ollama Service Status")
print("=" * 50)

try:
    # Check Ollama version
    response = requests.get("http://localhost:11434/api/version", timeout=2)
    version = response.json().get('version', 'Unknown')
    print(f"✓ Ollama Running: v{version}")
    print(f"Endpoint: http://localhost:11434")
    
    # List models
    result = subprocess.run(['ollama', 'list'], capture_output=True, text=True)
    print(f"\n📦 Installed Models:")
    print(result.stdout)
    
    # Check if llama2 is running
    ps_result = subprocess.run(['ps', 'aux'], capture_output=True, text=True)
    if 'llama' in ps_result.stdout.lower():
        print("✓ Llama2 model is currently loaded in memory")
    else:
        print("ℹ️ Llama2 will load on first request")
    
except requests.exceptions.RequestException:
    print("❌ Ollama service not running")
    print("Start with: brew services start ollama")
except FileNotFoundError:
    print("❌ Ollama not installed")
    print("Install with: brew install ollama")

## 4. OpenAI Configuration Check

Check if OpenAI API key is configured and test connection.

In [ ]:
# OpenAI Configuration Check
from src.utils.config import get_config

config = get_config()

print("🔑 OpenAI Configuration")
print("=" * 50)
print(f"Current LLM Provider: {config.llm.provider}")

# Check if OpenAI key is set
openai_key = os.getenv("OPENAI_API_KEY")
if openai_key:
    print(f"✓ OpenAI API Key: Configured (ends with ...{openai_key[-4:]})")
    
    # Test OpenAI connection (optional)
    try:
        from openai import OpenAI
        client = OpenAI(api_key=openai_key)
        # Simple test - list models
        models = client.models.list()
        print("✓ OpenAI API: Connection successful")
        print(f"Available models: {len(list(models))} models found")
    except Exception as e:
        print(f"⚠️ OpenAI API test failed: {e}")
else:
    print("❌ OpenAI API Key: Not configured")
    print("\nTo configure OpenAI:")
    print("1. Copy .env.example to .env")
    print("2. Add your key: OPENAI_API_KEY=sk-...")
    print("3. Change config.yaml: llm.provider = 'openai'")

print(f"\n📝 Configuration:")
print(f"  Model: {config.llm.openai.model}")
print(f"  Temperature: {config.llm.openai.temperature}")
print(f"  Max Tokens: {config.llm.openai.max_tokens}")

## 5. Streamlit & Conversation History

Check conversation logs and usage statistics.

In [ ]:
# Streamlit & Conversation Monitoring
print("💬 Conversation History")
print("=" * 50)

# Check conversation directory
conv_dir = Path("./data/conversations")
if conv_dir.exists():
    conv_files = list(conv_dir.glob("*.json"))
    print(f"Total Conversations Saved: {len(conv_files)}")
    
    if conv_files:
        # Show recent conversations
        conv_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        print(f"\n📁 Recent Conversations:")
        
        for conv_file in conv_files[:5]:
            with open(conv_file) as f:
                data = json.load(f)
                msg_count = len(data.get('messages', []))
                timestamp = data.get('conversation_id', 'Unknown')
                print(f"  - {conv_file.name}: {msg_count} messages")
        
        # Analyze latest conversation
        print(f"\n🔍 Latest Conversation Analysis:")
        with open(conv_files[0]) as f:
            latest = json.load(f)
            messages = latest.get('messages', [])
            user_msgs = [m for m in messages if m['role'] == 'user']
            assistant_msgs = [m for m in messages if m['role'] == 'assistant']
            
            print(f"  User Questions: {len(user_msgs)}")
            print(f"  AI Responses: {len(assistant_msgs)}")
            
            if assistant_msgs:
                # Check if sources were used
                with_sources = sum(1 for m in assistant_msgs if m.get('sources'))
                print(f"  Responses with Sources: {with_sources}/{len(assistant_msgs)}")
                
                # Show confidence levels
                confidences = [m.get('confidence') for m in assistant_msgs if m.get('confidence')]
                if confidences:
                    print(f"  Confidence Levels: {', '.join(confidences)}")
    else:
        print("No conversations saved yet")
else:
    print("⚠️ Conversations directory not found")

# Check logs
log_dir = Path("./logs")
if log_dir.exists():
    log_files = list(log_dir.glob("*.log"))
    if log_files:
        latest_log = max(log_files, key=lambda x: x.stat().st_mtime)
        log_size = latest_log.stat().st_size / 1024
        print(f"\n📋 Latest Log: {latest_log.name} ({log_size:.2f} KB)")

## 6. Document Ingestion Timeline

Track when documents were added to the system.

In [ ]:
# Document Ingestion Timeline
print("📚 Document Ingestion History")
print("=" * 50)

raw_dir = Path("./data/raw")
if raw_dir.exists():
    files = list(raw_dir.glob("*"))
    if files:
        # Create timeline
        file_data = []
        for f in files:
            if f.is_file():
                stat = f.stat()
                file_data.append({
                    'Filename': f.name,
                    'Size (KB)': stat.st_size / 1024,
                    'Modified': datetime.fromtimestamp(stat.st_mtime).strftime('%Y-%m-%d %H:%M:%S'),
                    'Type': f.suffix
                })
        
        if file_data:
            df = pd.DataFrame(file_data)
            df = df.sort_values('Modified', ascending=False)
            print(df.to_string(index=False))
            print(f"\n📊 Summary:")
            print(f"  Total Files: {len(df)}")
            print(f"  Total Size: {df['Size (KB)'].sum():.2f} KB")
            print(f"  File Types: {df['Type'].value_counts().to_dict()}")
    else:
        print("No documents uploaded yet")
else:
    print("⚠️ Raw data directory not found")

## 7. Real-time LLM Usage Test

Test which LLM is actually being used (Ollama vs OpenAI).

In [ ]:
# Real-time LLM Test
from src.rag.rag_pipeline import get_rag_pipeline
import time

print("🧪 Testing Active LLM")
print("=" * 50)

rag = get_rag_pipeline()

# Test query
test_query = "What is semiconductor testing?"
print(f"Query: {test_query}")
print("\nProcessing...")

start_time = time.time()
try:
    result = rag.query(test_query)
    elapsed = time.time() - start_time
    
    print(f"\n✓ Response received in {elapsed:.2f} seconds")
    print(f"\nAnswer: {result.answer[:200]}...")
    print(f"\n📊 Details:")
    print(f"  Sources Used: {len(result.sources)}")
    print(f"  Confidence: {result.confidence}")
    print(f"  Processing Time: {result.processing_time:.2f}s")
    
    # Check which LLM was used by checking Ollama logs
    print(f"\n🔍 LLM Used: {config.llm.provider}")
    if config.llm.provider == "ollama":
        print("  Model: llama2 (via Ollama)")
        print("  Cost: FREE")
    else:
        print(f"  Model: {config.llm.openai.model} (via OpenAI)")
        print("  Cost: ~$0.0015-0.002 per query")
    
except Exception as e:
    print(f"\n❌ Error: {e}")
    print("\nCheck:")
    print("  - Is Ollama running? (brew services list)")
    print("  - Is llama2 model installed? (ollama list)")
    print("  - Or set OpenAI key in .env file")

## 8. Complete System Health Check

Quick overview of all components.

In [ ]:
# System Health Dashboard
print("🏥 System Health Dashboard")
print("=" * 60)

health = {
    'Component': [],
    'Status': [],
    'Details': []
}

# 1. Embeddings
try:
    embeddings_mgr.embed_query("test")
    health['Component'].append('1. Embeddings')
    health['Status'].append('✅ Running')
    health['Details'].append('instructor-large on CPU')
except:
    health['Component'].append('1. Embeddings')
    health['Status'].append('❌ Failed')
    health['Details'].append('Check installation')

# 2. ChromaDB
try:
    stats = vectorstore.get_stats()
    health['Component'].append('2. ChromaDB')
    health['Status'].append('✅ Running')
    health['Details'].append(f"{stats.get('total_documents', 0)} chunks stored")
except:
    health['Component'].append('2. ChromaDB')
    health['Status'].append('❌ Failed')
    health['Details'].append('Check database')

# 3. Ollama
try:
    r = requests.get("http://localhost:11434/api/version", timeout=2)
    health['Component'].append('3. Ollama LLM')
    health['Status'].append('✅ Running')
    health['Details'].append(f"llama2 ready")
except:
    health['Component'].append('3. Ollama LLM')
    health['Status'].append('❌ Offline')
    health['Details'].append('Start: brew services start ollama')

# 4. Streamlit
try:
    r = requests.get("http://localhost:8501", timeout=2)
    health['Component'].append('4. Streamlit UI')
    health['Status'].append('✅ Running')
    health['Details'].append('http://localhost:8501')
except:
    health['Component'].append('4. Streamlit UI')
    health['Status'].append('❌ Offline')
    health['Details'].append('Start: streamlit run src/ui/app.py')

# 5. OpenAI
if os.getenv("OPENAI_API_KEY"):
    health['Component'].append('5. OpenAI (Optional)')
    health['Status'].append('✅ Configured')
    health['Details'].append('Available as fallback')
else:
    health['Component'].append('5. OpenAI (Optional)')
    health['Status'].append('⚪ Not Set')
    health['Details'].append('Optional - using Ollama')

df = pd.DataFrame(health)
print(df.to_string(index=False))

# Summary
running = sum(1 for s in health['Status'] if '✅' in s)
print(f"\n📊 Summary: {running}/{len(health['Status'])} components running")
print(f"\n⏰ Check completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")